In [ ]:
import torch 
import onnx 
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.export(
    format = "onnx",
    imgsz = 640,
    batch = 1,
    dynamic = False,
    simplify = False,
    opset = 17
)

print("ONNX export completed")

In [2]:
import onnx

model = onnx.load("yolov8n.onnx")

onnx.checker.check_model(model)

print("ONNX model is valid!")

print("\nInputs:")
for input_tensor in model.graph.input:
    print(input_tensor.name)

print("\nOutputs:")
for output_tensor in model.graph.output:
    print(output_tensor.name)

ONNX model is valid!

Inputs:
images

Outputs:
output0


In [8]:

import cv2
import numpy as np
import torch
import onnxruntime as ort
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox

IMAGE_PATH = "bus.jpg"
IMG_SIZE = 640
CONF = 0.25

# 1. Load original image
image = cv2.imread(IMAGE_PATH)

if image is None:
    raise RuntimeError("Could not load image")

# 2. Preprocess exactly once
letterbox = LetterBox(
    new_shape=(IMG_SIZE, IMG_SIZE),
    auto=False
)

processed = letterbox({"img": image})["img"]

# BGR -> RGB, HWC -> CHW
processed = processed[:, :, ::-1].transpose(2, 0, 1)
processed = np.ascontiguousarray(processed, dtype=np.float32)
processed /= 255.0

input_tensor = np.expand_dims(processed, axis=0)

print("Input shape:", input_tensor.shape)

print("Preprocessed image:", processed.shape)
print("ONNX input:", input_tensor.shape)

# 3. PyTorch inference
pytorch_model = YOLO("yolov8n.pt")

pytorch_results = pytorch_model.predict(
    source=IMAGE_PATH,
    imgsz=IMG_SIZE,
    conf=CONF,
    device=0,
    rect=False,
    verbose=False
)

pytorch_boxes = pytorch_results[0].boxes.xyxy.cpu().numpy()
pytorch_scores = pytorch_results[0].boxes.conf.cpu().numpy()

# 4. ONNX inference
session = ort.InferenceSession(
    "yolov8n.onnx",
    providers=["CPUExecutionProvider"]
)

onnx_output = session.run(
    None,
    {session.get_inputs()[0].name: input_tensor}
)

print("ONNX output shape:", onnx_output[0].shape)

# 5. Results
print("\n--- RESULTS ---")
print("PyTorch detections:", len(pytorch_boxes))
print("ONNX raw output:", onnx_output[0].shape)
print("PyTorch confidence scores:", pytorch_scores)

print("\nBoth inference processes completed.")


Input shape: (1, 3, 640, 640)
Preprocessed image: (3, 640, 640)
ONNX input: (1, 3, 640, 640)
ONNX output shape: (1, 84, 8400)

--- RESULTS ---
PyTorch detections: 5
ONNX raw output: (1, 84, 8400)
PyTorch confidence scores: [    0.89026     0.88316     0.87853     0.84312     0.43581]

Both inference processes completed.
